In [1]:
import json

def create_outcome_jsonl():
    input_file = "/home/jw161/chatbot/basic_rag_testing/testbench/mcQTSA_test_final.jsonl"
    output_file = "Outcome_1000_keyword.jsonl"
    
    try:
        with open(input_file, 'r', encoding='utf-8') as f_in:
            lines = f_in.readlines()

        if len(lines) < 1000:
            print(f"警告：输入文件只有 {len(lines)} 行，但需要100行")
            lines = lines[:1000] 
        
        with open(output_file, 'w', encoding='utf-8') as f_out:
            for i in range(1000):
                if i < len(lines):
                
                    original_data = json.loads(lines[i].strip())
                    answer = original_data.get("answer", "")
                else:
                    answer = ""

                new_data = {
                    "ID": str(i + 1),  
                    "Correct Answer": answer
                }

                f_out.write(json.dumps(new_data, ensure_ascii=False) + '\n')
        
        print(f"Finish set up {output_file} file，including {min(1000, len(lines))} data")
        
    except FileNotFoundError:
        print(f"错误：找不到输入文件 {input_file}")
    except json.JSONDecodeError as e:
        print(f"错误：JSON解析失败 - {e}")
    except Exception as e:
        print(f"错误：{e}")

if __name__ == "__main__":
    create_outcome_jsonl()

Finish set up Outcome_1000_keyword.jsonl file，including 1000 data


In [3]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "1000_GPT_4o_RAG_keyword.jsonl"
outcome_file_path = "Outcome_1000_keyword.jsonl"

def extract_answer_from_output(question, model_output):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{model_output}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    model_output = item.get('model_output', '')
    answer = extract_answer_from_output(question, model_output)

    outcome_data[i]["GPT 4o with Keyword RAG Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

ID 1: Extracted answer is B
ID 2: Extracted answer is B
ID 3: Extracted answer is C
ID 4: Extracted answer is B
ID 5: Extracted answer is C
ID 6: Extracted answer is B
ID 7: Extracted answer is B
ID 8: Extracted answer is B
ID 9: Extracted answer is A
ID 10: Extracted answer is C
ID 11: Extracted answer is A
ID 12: Extracted answer is B
ID 13: Extracted answer is B
ID 14: Extracted answer is B
ID 15: Extracted answer is B
ID 16: Extracted answer is A
ID 17: Extracted answer is B
ID 18: Extracted answer is C
ID 19: Extracted answer is B
ID 20: Extracted answer is B
ID 21: Extracted answer is C
ID 22: Extracted answer is C
ID 23: Extracted answer is C
ID 24: Extracted answer is B
ID 25: Extracted answer is C
ID 26: Extracted answer is B
ID 27: Extracted answer is A
ID 28: Extracted answer is B
ID 29: Extracted answer is B
ID 30: Extracted answer is B
ID 31: Extracted answer is B
ID 32: Extracted answer is C
ID 33: Extracted answer is B
ID 34: Extracted answer is A
ID 35: Extracted answer

In [5]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "300_Qwen2.5_0.5B_RAG.jsonl"
outcome_file_path = "Outcome_300.jsonl"

def extract_answer_from_output(question, model_output):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{model_output}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    model_output = item.get('model_output', '')
    answer = extract_answer_from_output(question, model_output)

    outcome_data[i]["Qwen2.5_0.5B_RAG Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

ID 1: Extracted answer is B
ID 2: Extracted answer is C
ID 3: Extracted answer is A
ID 4: Extracted answer is A
ID 5: Extracted answer is C
ID 6: Extracted answer is B
ID 7: Extracted answer is B
ID 8: Extracted answer is B
ID 9: Extracted answer is A
ID 10: Extracted answer is B
ID 11: Extracted answer is B
ID 12: Extracted answer is D
ID 13: Extracted answer is B
ID 14: Extracted answer is B
ID 15: Extracted answer is D
ID 16: Extracted answer is B
ID 17: Extracted answer is A
ID 18: Extracted answer is C
ID 19: Extracted answer is A
ID 20: Extracted answer is B
ID 21: Extracted answer is D
ID 22: Extracted answer is B
ID 23: Extracted answer is A
ID 24: Extracted answer is B
ID 25: Extracted answer is B
ID 26: Extracted answer is B
ID 27: Extracted answer is A
ID 28: Extracted answer is B
ID 29: Extracted answer is B
ID 30: Extracted answer is B
ID 31: Extracted answer is B
ID 32: Extracted answer is A
ID 33: Extracted answer is B
ID 34: Extracted answer is B
ID 35: Extracted answer

In [6]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "300_Qwen2.5_1.5B_Instruct_RAG.jsonl"
outcome_file_path = "Outcome_300.jsonl"

def extract_answer_from_output(question, model_output):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{model_output}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    model_output = item.get('model_output', '')
    answer = extract_answer_from_output(question, model_output)

    outcome_data[i]["Qwen2.5_1.5B_Instruct_RAG Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

ID 1: Extracted answer is B
ID 2: Extracted answer is C
ID 3: Extracted answer is A
ID 4: Extracted answer is B
ID 5: Extracted answer is A
ID 6: Extracted answer is B
ID 7: Extracted answer is B
ID 8: Extracted answer is B
ID 9: Extracted answer is A
ID 10: Extracted answer is B
ID 11: Extracted answer is B
ID 12: Extracted answer is D
ID 13: Extracted answer is B
ID 14: Extracted answer is B
ID 15: Extracted answer is B
ID 16: Extracted answer is A
ID 17: Extracted answer is A
ID 18: Extracted answer is C
ID 19: Extracted answer is D
ID 20: Extracted answer is B
ID 21: Extracted answer is A
ID 22: Extracted answer is C
ID 23: Extracted answer is C
ID 24: Extracted answer is B
ID 25: Extracted answer is C
ID 26: Extracted answer is B
ID 27: Extracted answer is A
ID 28: Extracted answer is B
ID 29: Extracted answer is B
ID 30: Extracted answer is B
ID 31: Extracted answer is B
ID 32: Extracted answer is A
ID 33: Extracted answer is B
ID 34: Extracted answer is B
ID 35: Extracted answer

In [7]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "300_Qwen2.5_3B_Instruct_RAG.jsonl"
outcome_file_path = "Outcome_300.jsonl"

def extract_answer_from_output(question, model_output):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{model_output}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    model_output = item.get('model_output', '')
    answer = extract_answer_from_output(question, model_output)

    outcome_data[i]["Qwen2.5_3B_Instruct_RAG Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

ID 1: Extracted answer is B
ID 2: Extracted answer is B
ID 3: Extracted answer is A
ID 4: Extracted answer is B
ID 5: Extracted answer is C
ID 6: Extracted answer is B
ID 7: Extracted answer is B
ID 8: Extracted answer is B
ID 9: Extracted answer is A
ID 10: Extracted answer is C
ID 11: Extracted answer is B
ID 12: Extracted answer is B
ID 13: Extracted answer is None
ID 14: Extracted answer is B
ID 15: Extracted answer is B
ID 16: Extracted answer is B
ID 17: Extracted answer is A
ID 18: Extracted answer is C
ID 19: Extracted answer is D
ID 20: Extracted answer is C
ID 21: Extracted answer is A
ID 22: Extracted answer is C
ID 23: Extracted answer is A
ID 24: Extracted answer is B
ID 25: Extracted answer is C
ID 26: Extracted answer is B
ID 27: Extracted answer is B
ID 28: Extracted answer is B
ID 29: Extracted answer is B
ID 30: Extracted answer is B
ID 31: Extracted answer is B
ID 32: Extracted answer is C
ID 33: Extracted answer is B
ID 34: Extracted answer is None
ID 35: Extracted 

In [8]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "300_Qwen2.5_3B_RAG.jsonl"
outcome_file_path = "Outcome_300.jsonl"

def extract_answer_from_output(question, model_output):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{model_output}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    model_output = item.get('model_output', '')
    answer = extract_answer_from_output(question, model_output)

    outcome_data[i]["Qwen2.5_3B_RAG Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

ID 1: Extracted answer is B
ID 2: Extracted answer is C
ID 3: Extracted answer is C
ID 4: Extracted answer is B
ID 5: Extracted answer is A
ID 6: Extracted answer is B
ID 7: Extracted answer is B
ID 8: Extracted answer is B
ID 9: Extracted answer is A
ID 10: Extracted answer is B
ID 11: Extracted answer is B
ID 12: Extracted answer is B
ID 13: Extracted answer is B
ID 14: Extracted answer is B
ID 15: Extracted answer is B
ID 16: Extracted answer is A
ID 17: Extracted answer is A
ID 18: Extracted answer is C
ID 19: Extracted answer is B
ID 20: Extracted answer is B
ID 21: Extracted answer is C
ID 22: Extracted answer is B
ID 23: Extracted answer is C
ID 24: Extracted answer is B
ID 25: Extracted answer is D
ID 26: Extracted answer is B
ID 27: Extracted answer is A
ID 28: Extracted answer is B
ID 29: Extracted answer is B
ID 30: Extracted answer is B
ID 31: Extracted answer is B
ID 32: Extracted answer is A
ID 33: Extracted answer is B
ID 34: Extracted answer is B
ID 35: Extracted answer

In [9]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "300_Qwen2.5_1.5B_RAG.jsonl"
outcome_file_path = "Outcome_300.jsonl"

def extract_answer_from_output(question, model_output):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{model_output}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    model_output = item.get('model_output', '')
    answer = extract_answer_from_output(question, model_output)

    outcome_data[i]["Qwen2.5_1.5B_RAG Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

ID 1: Extracted answer is B
ID 2: Extracted answer is B
ID 3: Extracted answer is C
ID 4: Extracted answer is D
ID 5: Extracted answer is A
ID 6: Extracted answer is B
ID 7: Extracted answer is B
ID 8: Extracted answer is B
ID 9: Extracted answer is A
ID 10: Extracted answer is C
ID 11: Extracted answer is A
ID 12: Extracted answer is B
ID 13: Extracted answer is B
ID 14: Extracted answer is B
ID 15: Extracted answer is B
ID 16: Extracted answer is A
ID 17: Extracted answer is None
ID 18: Extracted answer is C
ID 19: Extracted answer is B
ID 20: Extracted answer is B
ID 21: Extracted answer is D
ID 22: Extracted answer is B
ID 23: Extracted answer is None
ID 24: Extracted answer is B
ID 25: Extracted answer is B
ID 26: Extracted answer is B
ID 27: Extracted answer is B
ID 28: Extracted answer is B
ID 29: Extracted answer is B
ID 30: Extracted answer is B
ID 31: Extracted answer is B
ID 32: Extracted answer is C
ID 33: Extracted answer is B
ID 34: Extracted answer is A
ID 35: Extracted 

In [10]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "300_Qwen2.5_0.5B_finetuned_RAG.jsonl"
outcome_file_path = "Outcome_300.jsonl"

def extract_answer_from_output(question, model_output):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{model_output}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    model_output = item.get('model_output', '')
    answer = extract_answer_from_output(question, model_output)

    outcome_data[i]["Qwen2.5_0.5B_finetuned_RAG Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

ID 1: Extracted answer is B
ID 2: Extracted answer is D
ID 3: Extracted answer is B
ID 4: Extracted answer is B
ID 5: Extracted answer is A
ID 6: Extracted answer is B
ID 7: Extracted answer is B
ID 8: Extracted answer is B
ID 9: Extracted answer is A
ID 10: Extracted answer is A
ID 11: Extracted answer is B
ID 12: Extracted answer is B
ID 13: Extracted answer is B
ID 14: Extracted answer is D
ID 15: Extracted answer is B
ID 16: Extracted answer is D
ID 17: Extracted answer is B
ID 18: Extracted answer is B
ID 19: Extracted answer is D
ID 20: Extracted answer is B
ID 21: Extracted answer is D
ID 22: Extracted answer is C
ID 23: Extracted answer is C
ID 24: Extracted answer is C
ID 25: Extracted answer is C
ID 26: Extracted answer is B
ID 27: Extracted answer is A
ID 28: Extracted answer is B
ID 29: Extracted answer is B
ID 30: Extracted answer is B
ID 31: Extracted answer is B
ID 32: Extracted answer is C
ID 33: Extracted answer is B
ID 34: Extracted answer is B
ID 35: Extracted answer

In [11]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "300_Qwen2.5_0.5B_Instruct_finetuned_RAG.jsonl"
outcome_file_path = "Outcome_300.jsonl"

def extract_answer_from_output(question, model_output):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{model_output}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    model_output = item.get('model_output', '')
    answer = extract_answer_from_output(question, model_output)

    outcome_data[i]["Qwen2.5_0.5B_Instruct_finetuned_RAG Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

ID 1: Extracted answer is B
ID 2: Extracted answer is D
ID 3: Extracted answer is D
ID 4: Extracted answer is B
ID 5: Extracted answer is C
ID 6: Extracted answer is B
ID 7: Extracted answer is B
ID 8: Extracted answer is B
ID 9: Extracted answer is A
ID 10: Extracted answer is B
ID 11: Extracted answer is B
ID 12: Extracted answer is B
ID 13: Extracted answer is B
ID 14: Extracted answer is None
ID 15: Extracted answer is B
ID 16: Extracted answer is A
ID 17: Extracted answer is B
ID 18: Extracted answer is D
ID 19: Extracted answer is B
ID 20: Extracted answer is B
ID 21: Extracted answer is C
ID 22: Extracted answer is A
ID 23: Extracted answer is B
ID 24: Extracted answer is B
ID 25: Extracted answer is B
ID 26: Extracted answer is B
ID 27: Extracted answer is B
ID 28: Extracted answer is B
ID 29: Extracted answer is B
ID 30: Extracted answer is B
ID 31: Extracted answer is B
ID 32: Extracted answer is A
ID 33: Extracted answer is B
ID 34: Extracted answer is B
ID 35: Extracted ans

In [12]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "300_Qwen2.5_1.5B_Instruct_finetuned_RAG.jsonl"
outcome_file_path = "Outcome_300.jsonl"

def extract_answer_from_output(question, model_output):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{model_output}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    model_output = item.get('model_output', '')
    answer = extract_answer_from_output(question, model_output)

    outcome_data[i]["Qwen2.5_1.5B_Instruct_finetuned_RAG Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

ID 1: Extracted answer is B
ID 2: Extracted answer is B
ID 3: Extracted answer is C
ID 4: Extracted answer is B
ID 5: Extracted answer is A
ID 6: Extracted answer is B
ID 7: Extracted answer is B
ID 8: Extracted answer is B
ID 9: Extracted answer is B
ID 10: Extracted answer is B
ID 11: Extracted answer is B
ID 12: Extracted answer is B
ID 13: Extracted answer is B
ID 14: Extracted answer is B
ID 15: Extracted answer is B
ID 16: Extracted answer is A
ID 17: Extracted answer is A
ID 18: Extracted answer is A
ID 19: Extracted answer is B
ID 20: Extracted answer is B
ID 21: Extracted answer is A
ID 22: Extracted answer is B
ID 23: Extracted answer is A
ID 24: Extracted answer is B
ID 25: Extracted answer is B
ID 26: Extracted answer is B
ID 27: Extracted answer is A
ID 28: Extracted answer is B
ID 29: Extracted answer is B
ID 30: Extracted answer is B
ID 31: Extracted answer is B
ID 32: Extracted answer is D
ID 33: Extracted answer is B
ID 34: Extracted answer is None
ID 35: Extracted ans

In [13]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "300_Qwen2.5_1.5B_finetuned_RAG.jsonl"
outcome_file_path = "Outcome_300.jsonl"

def extract_answer_from_output(question, model_output):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{model_output}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    model_output = item.get('model_output', '')
    answer = extract_answer_from_output(question, model_output)

    outcome_data[i]["Qwen2.5_1.5B_finetuned_RAG Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

ID 1: Extracted answer is B
ID 2: Extracted answer is C
ID 3: Extracted answer is A
ID 4: Extracted answer is B
ID 5: Extracted answer is B
ID 6: Extracted answer is B
ID 7: Extracted answer is B
ID 8: Extracted answer is B
ID 9: Extracted answer is A
ID 10: Extracted answer is B
ID 11: Extracted answer is B
ID 12: Extracted answer is C
ID 13: Extracted answer is B
ID 14: Extracted answer is B
ID 15: Extracted answer is B
ID 16: Extracted answer is A
ID 17: Extracted answer is B
ID 18: Extracted answer is C
ID 19: Extracted answer is B
ID 20: Extracted answer is B
ID 21: Extracted answer is D
ID 22: Extracted answer is B
ID 23: Extracted answer is D
ID 24: Extracted answer is B
ID 25: Extracted answer is C
ID 26: Extracted answer is B
ID 27: Extracted answer is D
ID 28: Extracted answer is B
ID 29: Extracted answer is B
ID 30: Extracted answer is B
ID 31: Extracted answer is B
ID 32: Extracted answer is C
ID 33: Extracted answer is B
ID 34: Extracted answer is B
ID 35: Extracted answer

In [14]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "300_Qwen2.5_3B_finetuned_RAG.jsonl"
outcome_file_path = "Outcome_300.jsonl"

def extract_answer_from_output(question, model_output):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{model_output}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    model_output = item.get('model_output', '')
    answer = extract_answer_from_output(question, model_output)

    outcome_data[i]["Qwen2.5_3B_finetuned_RAG Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

ID 1: Extracted answer is None
ID 2: Extracted answer is A
ID 3: Extracted answer is A
ID 4: Extracted answer is B
ID 5: Extracted answer is C
ID 6: Extracted answer is B
ID 7: Extracted answer is B
ID 8: Extracted answer is B
ID 9: Extracted answer is A
ID 10: Extracted answer is C
ID 11: Extracted answer is B
ID 12: Extracted answer is B
ID 13: Extracted answer is B
ID 14: Extracted answer is B
ID 15: Extracted answer is B
ID 16: Extracted answer is A
ID 17: Extracted answer is A
ID 18: Extracted answer is C
ID 19: Extracted answer is B
ID 20: Extracted answer is C
ID 21: Extracted answer is D
ID 22: Extracted answer is B
ID 23: Extracted answer is A
ID 24: Extracted answer is B
ID 25: Extracted answer is C
ID 26: Extracted answer is B
ID 27: Extracted answer is B
ID 28: Extracted answer is B
ID 29: Extracted answer is B
ID 30: Extracted answer is B
ID 31: Extracted answer is B
ID 32: Extracted answer is C
ID 33: Extracted answer is B
ID 34: Extracted answer is None
ID 35: Extracted 

In [15]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "300_Qwen2.5_3B_Instruct_finetuned_RAG.jsonl"
outcome_file_path = "Outcome_300.jsonl"

def extract_answer_from_output(question, model_output):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{model_output}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    model_output = item.get('model_output', '')
    answer = extract_answer_from_output(question, model_output)

    outcome_data[i]["Qwen2.5_3B_Instruct_finetuned_RAG Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

ID 1: Extracted answer is B
ID 2: Extracted answer is B
ID 3: Extracted answer is B
ID 4: Extracted answer is B
ID 5: Extracted answer is C
ID 6: Extracted answer is B
ID 7: Extracted answer is B
ID 8: Extracted answer is B
ID 9: Extracted answer is A
ID 10: Extracted answer is C
ID 11: Extracted answer is AC
ID 12: Extracted answer is B
ID 13: Extracted answer is B
ID 14: Extracted answer is B
ID 15: Extracted answer is B
ID 16: Extracted answer is A
ID 17: Extracted answer is B
ID 18: Extracted answer is None
ID 19: Extracted answer is C
ID 20: Extracted answer is B
ID 21: Extracted answer is D
ID 22: Extracted answer is B
ID 23: Extracted answer is None
ID 24: Extracted answer is B
ID 25: Extracted answer is C
ID 26: Extracted answer is B
ID 27: Extracted answer is B
ID 28: Extracted answer is B
ID 29: Extracted answer is B
ID 30: Extracted answer is B
ID 31: Extracted answer is B
ID 32: Extracted answer is C
ID 33: Extracted answer is B
ID 34: Extracted answer is B
ID 35: Extracted

In [16]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "300_Qwen3_0.6B_thinking_RAG.jsonl"
outcome_file_path = "Outcome_300.jsonl"

def extract_answer_from_output(question, model_output):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{model_output}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    model_output = item.get('model_output', '')
    answer = extract_answer_from_output(question, model_output)

    outcome_data[i]["Qwen3_0.6B_thinking_RAG Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

ID 1: Extracted answer is B
ID 2: Extracted answer is B
ID 3: Extracted answer is C
ID 4: Extracted answer is B
ID 5: Extracted answer is C
ID 6: Extracted answer is B
ID 7: Extracted answer is B
ID 8: Extracted answer is B
ID 9: Extracted answer is A
ID 10: Extracted answer is C
ID 11: Extracted answer is B
ID 12: Extracted answer is A
ID 13: Extracted answer is B
ID 14: Extracted answer is B
ID 15: Extracted answer is B
ID 16: Extracted answer is A
ID 17: Extracted answer is A
ID 18: Extracted answer is C
ID 19: Extracted answer is D
ID 20: Extracted answer is A
ID 21: Extracted answer is A
ID 22: Extracted answer is B
ID 23: Extracted answer is C
ID 24: Extracted answer is B
ID 25: Extracted answer is B
ID 26: Extracted answer is B
ID 27: Extracted answer is A
ID 28: Extracted answer is B
ID 29: Extracted answer is B
ID 30: Extracted answer is B
ID 31: Extracted answer is B
ID 32: Extracted answer is B
ID 33: Extracted answer is B
ID 34: Extracted answer is B
ID 35: Extracted answer

In [17]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "300_Qwen3_0.6B_nothinking_RAG.jsonl"
outcome_file_path = "Outcome_300.jsonl"

def extract_answer_from_output(question, model_output):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{model_output}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    model_output = item.get('model_output', '')
    answer = extract_answer_from_output(question, model_output)

    outcome_data[i]["Qwen3_0.6B_nothinking_RAG Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

ID 1: Extracted answer is B
ID 2: Extracted answer is B
ID 3: Extracted answer is A
ID 4: Extracted answer is B
ID 5: Extracted answer is C
ID 6: Extracted answer is B
ID 7: Extracted answer is B
ID 8: Extracted answer is A
ID 9: Extracted answer is A
ID 10: Extracted answer is C
ID 11: Extracted answer is B
ID 12: Extracted answer is B
ID 13: Extracted answer is B
ID 14: Extracted answer is B
ID 15: Extracted answer is B
ID 16: Extracted answer is A
ID 17: Extracted answer is A
ID 18: Extracted answer is C
ID 19: Extracted answer is B
ID 20: Extracted answer is C
ID 21: Extracted answer is A
ID 22: Extracted answer is B
ID 23: Extracted answer is A
ID 24: Extracted answer is B
ID 25: Extracted answer is C
ID 26: Extracted answer is B
ID 27: Extracted answer is A
ID 28: Extracted answer is B
ID 29: Extracted answer is B
ID 30: Extracted answer is A
ID 31: Extracted answer is B
ID 32: Extracted answer is B
ID 33: Extracted answer is B
ID 34: Extracted answer is D
ID 35: Extracted answer

In [18]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "300_Qwen3_1.7B_thinking_RAG.jsonl"
outcome_file_path = "Outcome_300.jsonl"

def extract_answer_from_output(question, model_output):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{model_output}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    model_output = item.get('model_output', '')
    answer = extract_answer_from_output(question, model_output)

    outcome_data[i]["Qwen3_1.7B_thinking_RAG Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

ID 1: Extracted answer is B
ID 2: Extracted answer is B
ID 3: Extracted answer is B
ID 4: Extracted answer is B
ID 5: Extracted answer is C
ID 6: Extracted answer is B
ID 7: Extracted answer is D
ID 8: Extracted answer is B
ID 9: Extracted answer is A
ID 10: Extracted answer is C
ID 11: Extracted answer is A
ID 12: Extracted answer is B
ID 13: Extracted answer is B
ID 14: Extracted answer is B
ID 15: Extracted answer is B
ID 16: Extracted answer is C
ID 17: Extracted answer is B
ID 18: Extracted answer is C
ID 19: Extracted answer is B
ID 20: Extracted answer is C
ID 21: Extracted answer is A
ID 22: Extracted answer is C
ID 23: Extracted answer is C
ID 24: Extracted answer is B
ID 25: Extracted answer is C
ID 26: Extracted answer is B
ID 27: Extracted answer is A
ID 28: Extracted answer is B
ID 29: Extracted answer is B
ID 30: Extracted answer is B
ID 31: Extracted answer is B
ID 32: Extracted answer is A
ID 33: Extracted answer is B
ID 34: Extracted answer is A
ID 35: Extracted answer

In [19]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "300_Qwen3_1.7B_nothinking_RAG.jsonl"
outcome_file_path = "Outcome_300.jsonl"

def extract_answer_from_output(question, model_output):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{model_output}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    model_output = item.get('model_output', '')
    answer = extract_answer_from_output(question, model_output)

    outcome_data[i]["Qwen3_1.7B_nothinking_RAG Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

ID 1: Extracted answer is B
ID 2: Extracted answer is A
ID 3: Extracted answer is A
ID 4: Extracted answer is B
ID 5: Extracted answer is C
ID 6: Extracted answer is B
ID 7: Extracted answer is D
ID 8: Extracted answer is B
ID 9: Extracted answer is A
ID 10: Extracted answer is C
ID 11: Extracted answer is A
ID 12: Extracted answer is B
ID 13: Extracted answer is B
ID 14: Extracted answer is B
ID 15: Extracted answer is B
ID 16: Extracted answer is C
ID 17: Extracted answer is B
ID 18: Extracted answer is C
ID 19: Extracted answer is D
ID 20: Extracted answer is A
ID 21: Extracted answer is A
ID 22: Extracted answer is C
ID 23: Extracted answer is C
ID 24: Extracted answer is B
ID 25: Extracted answer is C
ID 26: Extracted answer is B
ID 27: Extracted answer is A
ID 28: Extracted answer is B
ID 29: Extracted answer is B
ID 30: Extracted answer is B
ID 31: Extracted answer is B
ID 32: Extracted answer is A
ID 33: Extracted answer is B
ID 34: Extracted answer is A
ID 35: Extracted answer

In [20]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "300_Qwen3_4B_nothinking_RAG.jsonl"
outcome_file_path = "Outcome_300.jsonl"

def extract_answer_from_output(question, model_output):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{model_output}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    model_output = item.get('model_output', '')
    answer = extract_answer_from_output(question, model_output)

    outcome_data[i]["Qwen3_4B_nothinking_RAG Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

ID 1: Extracted answer is B
ID 2: Extracted answer is C
ID 3: Extracted answer is A
ID 4: Extracted answer is B
ID 5: Extracted answer is C
ID 6: Extracted answer is B
ID 7: Extracted answer is B
ID 8: Extracted answer is B
ID 9: Extracted answer is A
ID 10: Extracted answer is C
ID 11: Extracted answer is A
ID 12: Extracted answer is A
ID 13: Extracted answer is B
ID 14: Extracted answer is B
ID 15: Extracted answer is B
ID 16: Extracted answer is C
ID 17: Extracted answer is B
ID 18: Extracted answer is C
ID 19: Extracted answer is B
ID 20: Extracted answer is C
ID 21: Extracted answer is A
ID 22: Extracted answer is C
ID 23: Extracted answer is C
ID 24: Extracted answer is B
ID 25: Extracted answer is C
ID 26: Extracted answer is B
ID 27: Extracted answer is B
ID 28: Extracted answer is B
ID 29: Extracted answer is B
ID 30: Extracted answer is B
ID 31: Extracted answer is B
ID 32: Extracted answer is A
ID 33: Extracted answer is B
ID 34: Extracted answer is A
ID 35: Extracted answer

In [21]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "300_Qwen3_4B_thinking_RAG.jsonl"
outcome_file_path = "Outcome_300.jsonl"

def extract_answer_from_output(question, model_output):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{model_output}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    model_output = item.get('model_output', '')
    answer = extract_answer_from_output(question, model_output)

    outcome_data[i]["Qwen3_4B_thinking_RAG Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

ID 1: Extracted answer is B
ID 2: Extracted answer is C
ID 3: Extracted answer is A
ID 4: Extracted answer is B
ID 5: Extracted answer is C
ID 6: Extracted answer is B
ID 7: Extracted answer is B
ID 8: Extracted answer is B
ID 9: Extracted answer is A
ID 10: Extracted answer is C
ID 11: Extracted answer is A
ID 12: Extracted answer is A
ID 13: Extracted answer is B
ID 14: Extracted answer is B
ID 15: Extracted answer is B
ID 16: Extracted answer is C
ID 17: Extracted answer is B
ID 18: Extracted answer is C
ID 19: Extracted answer is B
ID 20: Extracted answer is C
ID 21: Extracted answer is D
ID 22: Extracted answer is C
ID 23: Extracted answer is C
ID 24: Extracted answer is B
ID 25: Extracted answer is C
ID 26: Extracted answer is B
ID 27: Extracted answer is B
ID 28: Extracted answer is B
ID 29: Extracted answer is B
ID 30: Extracted answer is B
ID 31: Extracted answer is B
ID 32: Extracted answer is A
ID 33: Extracted answer is B
ID 34: Extracted answer is A
ID 35: Extracted answer

In [22]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "300_Qwen3_4B_finetuned_thinking_RAG.jsonl"
outcome_file_path = "Outcome_300.jsonl"

def extract_answer_from_output(question, model_output):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{model_output}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    model_output = item.get('model_output', '')
    answer = extract_answer_from_output(question, model_output)

    outcome_data[i]["Qwen3_4B_finetuned_thinking_RAG Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

ID 1: Extracted answer is B
ID 2: Extracted answer is B
ID 3: Extracted answer is A
ID 4: Extracted answer is B
ID 5: Extracted answer is C
ID 6: Extracted answer is B
ID 7: Extracted answer is B
ID 8: Extracted answer is B
ID 9: Extracted answer is A
ID 10: Extracted answer is C
ID 11: Extracted answer is B
ID 12: Extracted answer is B
ID 13: Extracted answer is B
ID 14: Extracted answer is B
ID 15: Extracted answer is B
ID 16: Extracted answer is A
ID 17: Extracted answer is B
ID 18: Extracted answer is C
ID 19: Extracted answer is B
ID 20: Extracted answer is C
ID 21: Extracted answer is None
ID 22: Extracted answer is B
ID 23: Extracted answer is C
ID 24: Extracted answer is B
ID 25: Extracted answer is C
ID 26: Extracted answer is B
ID 27: Extracted answer is B
ID 28: Extracted answer is B
ID 29: Extracted answer is B
ID 30: Extracted answer is B
ID 31: Extracted answer is B
ID 32: Extracted answer is C
ID 33: Extracted answer is B
ID 34: Extracted answer is B
ID 35: Extracted ans

In [23]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "300_Qwen3_4B_finetuned_nothinking_RAG.jsonl"
outcome_file_path = "Outcome_300.jsonl"

def extract_answer_from_output(question, model_output):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{model_output}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    model_output = item.get('model_output', '')
    answer = extract_answer_from_output(question, model_output)

    outcome_data[i]["Qwen3_4B_finetuned_nothinking_RAG Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

ID 1: Extracted answer is B
ID 2: Extracted answer is B
ID 3: Extracted answer is A
ID 4: Extracted answer is B
ID 5: Extracted answer is C
ID 6: Extracted answer is B
ID 7: Extracted answer is B
ID 8: Extracted answer is B
ID 9: Extracted answer is A
ID 10: Extracted answer is C
ID 11: Extracted answer is D
ID 12: Extracted answer is B
ID 13: Extracted answer is B
ID 14: Extracted answer is B
ID 15: Extracted answer is B
ID 16: Extracted answer is A
ID 17: Extracted answer is B
ID 18: Extracted answer is C
ID 19: Extracted answer is B
ID 20: Extracted answer is B
ID 21: Extracted answer is A
ID 22: Extracted answer is B
ID 23: Extracted answer is C
ID 24: Extracted answer is B
ID 25: Extracted answer is B
ID 26: Extracted answer is B
ID 27: Extracted answer is B
ID 28: Extracted answer is B
ID 29: Extracted answer is B
ID 30: Extracted answer is B
ID 31: Extracted answer is B
ID 32: Extracted answer is A
ID 33: Extracted answer is B
ID 34: Extracted answer is B
ID 35: Extracted answer

In [25]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "300_Qwen3_1.7B_finetuned_nothinking_RAG.jsonl"
outcome_file_path = "Outcome_300.jsonl"

def extract_answer_from_output(question, model_output):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{model_output}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    model_output = item.get('model_output', '')
    answer = extract_answer_from_output(question, model_output)

    outcome_data[i]["Qwen3_1.7B_finetuned_nothinking_RAG Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

ID 1: Extracted answer is B
ID 2: Extracted answer is B
ID 3: Extracted answer is B
ID 4: Extracted answer is B
ID 5: Extracted answer is C
ID 6: Extracted answer is B
ID 7: Extracted answer is B
ID 8: Extracted answer is B
ID 9: Extracted answer is C
ID 10: Extracted answer is C
ID 11: Extracted answer is C
ID 12: Extracted answer is B
ID 13: Extracted answer is None
ID 14: Extracted answer is B
ID 15: Extracted answer is B
ID 16: Extracted answer is A
ID 17: Extracted answer is B
ID 18: Extracted answer is C
ID 19: Extracted answer is B
ID 20: Extracted answer is B
ID 21: Extracted answer is A
ID 22: Extracted answer is B
ID 23: Extracted answer is C
ID 24: Extracted answer is B
ID 25: Extracted answer is C
ID 26: Extracted answer is B
ID 27: Extracted answer is B
ID 28: Extracted answer is B
ID 29: Extracted answer is None
ID 30: Extracted answer is B
ID 31: Extracted answer is B
ID 32: Extracted answer is A
ID 33: Extracted answer is B
ID 34: Extracted answer is B
ID 35: Extracted 

In [26]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "300_Qwen3_1.7B_finetuned_thinking_RAG.jsonl"
outcome_file_path = "Outcome_300.jsonl"

def extract_answer_from_output(question, model_output):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{model_output}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    model_output = item.get('model_output', '')
    answer = extract_answer_from_output(question, model_output)

    outcome_data[i]["Qwen3_1.7B_finetuned_thinking_RAG Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

ID 1: Extracted answer is B
ID 2: Extracted answer is C
ID 3: Extracted answer is B
ID 4: Extracted answer is B
ID 5: Extracted answer is C
ID 6: Extracted answer is B
ID 7: Extracted answer is B
ID 8: Extracted answer is B
ID 9: Extracted answer is A
ID 10: Extracted answer is B
ID 11: Extracted answer is A
ID 12: Extracted answer is B
ID 13: Extracted answer is B
ID 14: Extracted answer is B
ID 15: Extracted answer is B
ID 16: Extracted answer is A
ID 17: Extracted answer is B
ID 18: Extracted answer is None
ID 19: Extracted answer is B
ID 20: Extracted answer is B
ID 21: Extracted answer is A
ID 22: Extracted answer is C
ID 23: Extracted answer is None
ID 24: Extracted answer is B
ID 25: Extracted answer is C
ID 26: Extracted answer is B
ID 27: Extracted answer is B
ID 28: Extracted answer is B
ID 29: Extracted answer is B
ID 30: Extracted answer is B
ID 31: Extracted answer is B
ID 32: Extracted answer is C
ID 33: Extracted answer is B
ID 34: Extracted answer is A
ID 35: Extracted 

In [27]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "300_Qwen3_0.6B_finetuned_thinking_RAG.jsonl"
outcome_file_path = "Outcome_300.jsonl"

def extract_answer_from_output(question, model_output):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{model_output}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    model_output = item.get('model_output', '')
    answer = extract_answer_from_output(question, model_output)

    outcome_data[i]["Qwen3_0.6B_finetuned_thinking_RAG Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

ID 1: Extracted answer is B
ID 2: Extracted answer is B
ID 3: Extracted answer is B
ID 4: Extracted answer is C
ID 5: Extracted answer is A
ID 6: Extracted answer is B
ID 7: Extracted answer is B
ID 8: Extracted answer is B
ID 9: Extracted answer is B
ID 10: Extracted answer is A
ID 11: Extracted answer is D
ID 12: Extracted answer is B
ID 13: Extracted answer is B
ID 14: Extracted answer is B
ID 15: Extracted answer is B
ID 16: Extracted answer is A
ID 17: Extracted answer is B
ID 18: Extracted answer is C
ID 19: Extracted answer is B
ID 20: Extracted answer is B
ID 21: Extracted answer is A
ID 22: Extracted answer is B
ID 23: Extracted answer is C
ID 24: Extracted answer is B
ID 25: Extracted answer is B
ID 26: Extracted answer is B
ID 27: Extracted answer is C
ID 28: Extracted answer is B
ID 29: Extracted answer is B
ID 30: Extracted answer is B
ID 31: Extracted answer is B
ID 32: Extracted answer is C
ID 33: Extracted answer is B
ID 34: Extracted answer is A
ID 35: Extracted answer

In [28]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "300_Qwen3_0.6B_finetuned_nothinking_RAG.jsonl"
outcome_file_path = "Outcome_300.jsonl"

def extract_answer_from_output(question, model_output):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{model_output}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    model_output = item.get('model_output', '')
    answer = extract_answer_from_output(question, model_output)

    outcome_data[i]["Qwen3_0.6B_finetuned_nothinking_RAG Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

ID 1: Extracted answer is B
ID 2: Extracted answer is None
ID 3: Extracted answer is B
ID 4: Extracted answer is B
ID 5: Extracted answer is A
ID 6: Extracted answer is B
ID 7: Extracted answer is B
ID 8: Extracted answer is B
ID 9: Extracted answer is A
ID 10: Extracted answer is C
ID 11: Extracted answer is A
ID 12: Extracted answer is B
ID 13: Extracted answer is B
ID 14: Extracted answer is B
ID 15: Extracted answer is B
ID 16: Extracted answer is A
ID 17: Extracted answer is A
ID 18: Extracted answer is A
ID 19: Extracted answer is B
ID 20: Extracted answer is B
ID 21: Extracted answer is A
ID 22: Extracted answer is B
ID 23: Extracted answer is C
ID 24: Extracted answer is B
ID 25: Extracted answer is C
ID 26: Extracted answer is B
ID 27: Extracted answer is B
ID 28: Extracted answer is B
ID 29: Extracted answer is B
ID 30: Extracted answer is B
ID 31: Extracted answer is B
ID 32: Extracted answer is C
ID 33: Extracted answer is B
ID 34: Extracted answer is None
ID 35: Extracted 